> **Research note:** Notebook prose may describe earlier exploratory analyses. Use regenerated artifacts and the current README for the implemented methodology.

# Portfolio optimization

The allocation below uses the same point-in-time portfolio function as the walk-forward backtest. It combines trailing historical returns, dated out-of-sample direction probabilities, MST centrality and a volatility-based cash overlay. The probability tilt is an uncalibrated research assumption. Synthetic sentiment is excluded. No CVaR constraint is claimed.


In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))
import numpy as np
import pandas as pd
from configs.settings import DATA_PROC, TRADING_DAYS_PER_YEAR
from src.io.loaders import load_log_returns, load_master_signals
from src.portfolio import construct_portfolio, trailing_signal

returns = load_log_returns().sort_index()
signals = load_master_signals()
as_of = returns.index[-1]
window = returns.loc[as_of - pd.DateOffset(months=12):as_of]
probabilities, signal_dates = trailing_signal(signals, as_of, returns.index, returns.columns)
weights, metadata = construct_portfolio(window, probabilities)
print(f"Allocation as of {as_of.date()}, regime {metadata['regime']}, cash {metadata['cash_pct']:.1%}")


In [ ]:
# The artifact uses the same portfolio function and inputs as notebook 12.
expected_vol = window.std() * np.sqrt(TRADING_DAYS_PER_YEAR)
out = pd.DataFrame({'Date': as_of, 'ticker': returns.columns, 'weight': weights.values,
                    'mu_combined': [metadata['mean_daily'][t] * TRADING_DAYS_PER_YEAR for t in returns.columns],
                    'expected_vol': expected_vol.reindex(returns.columns).values,
                    'signal_date': signal_dates.reindex(returns.columns).values})
out.to_csv(DATA_PROC / 'optimal_weights.csv', index=False)
out[['ticker', 'weight']].to_csv(DATA_PROC / 'optimized_weights.csv', index=False)
with open(DATA_PROC / 'portfolio_metrics.json', 'w') as f:
    json.dump({'as_of': str(as_of.date()), 'regime': metadata['regime'],
               'cash_pct': metadata['cash_pct'], 'signal_scale_daily': metadata['signal_scale_daily'],
               'method': 'shared point-in-time optimizer', 'sentiment_used': False}, f, indent=2)
print(out.sort_values('weight', ascending=False).head(10).to_string(index=False))
